In [25]:
import seaborn as sns
import pandas as pd
import numpy as np
import re

#orginele dataset inlezen
df = pd.read_csv('Leesmateriaal.csv')
df.head()

,Titel,Auteur,Korte omschrijving,Unnamed: 3,Type materiaal,Niveau,Themas_boek,materiaal_id,genre,onderwerp,...,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,interne_batchcode,administratieve_prioriteit,doelgroep_extra,genre_kopie,Gemiddelde beoordeling
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,NaN,boek,3F,WOII;onderduik,LM-0001,fictie,WOII,...,VERHALEN,Educatief Media,2025,PDF,CAT-00001,B1,Hoog,NaN,fictie,3.4
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,NaN,Boek,3F,spanning;stalking,LM-0002,fictie,spanning,...,verhalen,Leeswereld,2005,Digitaal,CAT-00002,B2,hoog,NT2,fictie,2.5
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",NaN,Boek,2F,liefde;relaties,LM-0003,fictie,liefde,...,verhalen,NaN,2025,Digitaal,CAT-00003,B2,Midden,NT2,fictie,3.8
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,NaN,Boek,3F,ontmoeting;verbondenheid,LM-0004,fictie,ontmoeting,...,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004,batch-2,Laag,NT2,fictie,3.9
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,NaN,Boek,2F,geheimen;schuld,LM-0005,fictie,geheimen,...,verhalen,Uitgeverij Noord,NaN,online,CAT-00005,B3,Laag,MBO/NT2,fictie,3.1


In [ ]:
#Analyse van de kolommen Themas_boek, trefwoorden en bron_url

def normaliseer_waarde(waarde):
    if pd.isna(waarde):
        return ""

    waarde = str(waarde).strip().casefold()

    # Spaties rondom puntkomma's negeren
    waarde = re.sub(r"\s*;\s*", ";", waarde)

    # Meerdere spaties vervangen door één spatie
    waarde = re.sub(r"\s+", " ", waarde)

    return waarde

# Waarden normaliseren
themas = df["Themas_boek"].map(normaliseer_waarde)
trefwoorden = df["trefwoorden"].map(normaliseer_waarde)
bron_url = df["bron_url"].map(normaliseer_waarde)

# Eerst bepalen of Themas_boek een URL bevat
is_url = themas.str.startswith(("http://", "https://"))

# Als Themas_boek een URL is, vergelijk met bron_url.
# Anders vergelijk met trefwoorden.
verwachte_waarde = trefwoorden.copy()
verwachte_waarde.loc[is_url] = bron_url.loc[is_url]

# Controleren welke rijen niet overeenkomen
verschilt = themas != verwachte_waarde

# Alleen afwijkende rijen weergeven
verschillen = df.loc[
    verschilt,
    [
        "Titel",
        "Themas_boek",
        "trefwoorden",
        "bron_url",
    ],
].copy()

# Laat zien waarmee Themas_boek per rij is vergeleken
verschillen["vergeleken_met"] = is_url.loc[verschillen.index].map(
    {
        True: "bron_url",
        False: "trefwoorden",
    }
)

if verschillen.empty:
    print("Alle waarden komen overeen.")
else:
    display(verschillen)

In [ ]:
#Analyse van de kolommen onderwerp en trefwoorden

def onderwerp_in_trefwoorden(rij):
    onderwerp = normaliseer_waarde(rij["onderwerp"])
    trefwoorden = normaliseer_waarde(rij["trefwoorden"])

    # Lege onderwerpen niet tonen als verschil
    if not onderwerp:
        return True

    # Trefwoorden opsplitsen in losse waarden
    trefwoord_lijst = trefwoorden.split(";")

    # Controleren of het onderwerp exact voorkomt als trefwoord
    return onderwerp in trefwoord_lijst

# Controleren per rij
onderwerp_ontbreekt = ~df.apply(onderwerp_in_trefwoorden, axis=1)

# Alleen rijen tonen waarin onderwerp niet in trefwoorden staat
onderwerp_verschillen = df.loc[
    onderwerp_ontbreekt,
    [
        "materiaal_id",
        "Titel",
        "onderwerp",
        "trefwoorden",
    ],
].copy()

if onderwerp_verschillen.empty:
    print("Elk onderwerp staat in de bijbehorende trefwoorden.")
else:
    display(onderwerp_verschillen)

In [ ]:
#Analyse van de kolommen genre en genre_kopie

# Waarden uit beide kolommen ophalen en normaliseren
genre = df["genre"].map(normaliseer_waarde)
genre_kopie = df["genre_kopie"].map(normaliseer_waarde)

# Alleen vergelijken als genre_kopie oorspronkelijk geen NaN is
heeft_genre_kopie = df["genre_kopie"].notna()

# Verschillen bepalen
verschilt_genre = (
    heeft_genre_kopie
    & (genre != genre_kopie)
)

# Alleen afwijkende rijen weergeven
genre_verschillen = df.loc[
    verschilt_genre,
    [
        "Titel",
        "genre",
        "genre_kopie",
    ],
].copy()

if genre_verschillen.empty:
    print("Alle ingevulde waarden in genre en genre_kopie komen overeen.")
else:
    display(genre_verschillen)

In [26]:
#Alle overige kolommen verwijderen die niet relevant zijn voor de analyse
df = df.drop(columns=['Unnamed: 3', 'materiaal_id', 'onderwerp', 'Themas_boek', 
                      'woordenaantal_omschrijving', 'gem_woordlengte', 'gem_zinslengte_woorden', 
                      'aandeel_lange_woorden_pct', 'aantal_zinnen', 'interne_batchcode', 
                      'administratieve_prioriteit', 'genre_kopie', 'Gemiddelde beoordeling'])

df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,doelgroep_extra
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,fictie,WOII; onderduik,Nederlands,265.0,556.0,NaN,MBO,Aangevuld (deels gesimuleerd),NaN,VERHALEN,Educatief Media,2025,PDF,CAT-00001,NaN
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,fictie,spanning; stalking,Nederlands,171.0,184.0,NaN,MBO,Aangevuld (deels gesimuleerd),48.8,verhalen,Leeswereld,2005,Digitaal,CAT-00002,NT2
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,fictie,liefde; relaties,Nederlands,219.0,662.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,NaN,2025,Digitaal,CAT-00003,NT2
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,fictie,ontmoeting; verbondenheid,Nederlands,288.0,748.0,NaN,MBO,Aangevuld (deels gesimuleerd),55.3,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004,NT2
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,fictie,geheimen; schuld,Nederlands,358.0,196.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,Uitgeverij Noord,NaN,online,CAT-00005,MBO/NT2


In [11]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 20)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,str,6
Auteur,231,7,str,45
Korte omschrijving,282,0,str,1
Type materiaal,10,1,str,272
Niveau,4,6,str,273
genre,3,26,str,254
trefwoorden,93,181,str,9
taal,4,0,str,279
aantal_paginas,159,17,float64,107
geschatte_leestijd_min,212,9,float64,62


In [27]:
#Merge de kolommen 'doelgroep' en 'doelgroep_extra' tot één kolom 'doelgroep', waarbij dubbele waarden worden verwijderd
def merge_doelgroep(row):
    unieke_doelgroepen = []

    for kolom in ["doelgroep", "doelgroep_extra"]:
        waarde = row[kolom]

        if pd.notna(waarde):
            for doelgroep in str(waarde).split("/"):
                doelgroep = doelgroep.strip()

                if doelgroep and doelgroep not in unieke_doelgroepen:
                    unieke_doelgroepen.append(doelgroep)

    return "/".join(unieke_doelgroepen)


df["doelgroep"] = df.apply(merge_doelgroep, axis=1)
df = df.drop(columns=["doelgroep_extra"])
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,fictie,WOII; onderduik,Nederlands,265.0,556.0,NaN,MBO,Aangevuld (deels gesimuleerd),NaN,VERHALEN,Educatief Media,2025,PDF,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,fictie,spanning; stalking,Nederlands,171.0,184.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),48.8,verhalen,Leeswereld,2005,Digitaal,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,fictie,liefde; relaties,Nederlands,219.0,662.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,NaN,2025,Digitaal,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,fictie,ontmoeting; verbondenheid,Nederlands,288.0,748.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),55.3,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,fictie,geheimen; schuld,Nederlands,358.0,196.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,Uitgeverij Noord,NaN,online,CAT-00005


In [28]:
#Missing values in de "Auteur" kolom vullen met "Onbekend"
df["Auteur"] = df["Auteur"].fillna("Onbekend")
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,fictie,WOII; onderduik,Nederlands,265.0,556.0,NaN,MBO,Aangevuld (deels gesimuleerd),NaN,VERHALEN,Educatief Media,2025,PDF,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,fictie,spanning; stalking,Nederlands,171.0,184.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),48.8,verhalen,Leeswereld,2005,Digitaal,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,fictie,liefde; relaties,Nederlands,219.0,662.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,NaN,2025,Digitaal,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,fictie,ontmoeting; verbondenheid,Nederlands,288.0,748.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),55.3,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,fictie,geheimen; schuld,Nederlands,358.0,196.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,Uitgeverij Noord,NaN,online,CAT-00005


In [5]:
lege_rijen = df[
    df["Type materiaal"].isna() |
    df["Type materiaal"].astype(str).str.strip().eq("")
]

lege_rijen.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
67,Het geluid van de stilte,Inez Weski,"Persoonlijk relaas over rechtspraak, media-aan...",NaN,3F+,NaN,NaN,Nederlands,200.0,617.0,NaN,MBO,NaN,54.0,verhalen,Leeswereld,NaN,Papier,CAT-00068


In [29]:
#in de "type materiaal" mist er 1 waarde waarvan we op hebben gezocht wat het is en dat invullen
df["Type materiaal"] = df["Type materiaal"].fillna("Boek")
df.head(70)

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,fictie,WOII; onderduik,Nederlands,265.0,556.0,NaN,MBO,Aangevuld (deels gesimuleerd),NaN,VERHALEN,Educatief Media,2025,PDF,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,fictie,spanning; stalking,Nederlands,171.0,184.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),48.8,verhalen,Leeswereld,2005,Digitaal,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,fictie,liefde; relaties,Nederlands,219.0,662.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,NaN,2025,Digitaal,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,fictie,ontmoeting; verbondenheid,Nederlands,288.0,748.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),55.3,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,fictie,geheimen; schuld,Nederlands,358.0,196.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),49.2,verhalen,Uitgeverij Noord,NaN,online,CAT-00005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Moordonderzoek,Malcolm Rose,Interactieve detective waarin de lezer aanwijz...,Boek,2F,fictie,NaN,Nederlands,NaN,604.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),65.9,verhalen,Uitgeverij Noord,2013,PDF,CAT-00066
66,Zonder spijt,Pieter Aspe,Spannende detectiveroman waarin een onderzoek ...,Boek - roman,3F,fictie,NaN,Nederlands,113.0,129.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),54.3,verhalen,NaN,2018,PDF,CAT-00067
67,Het geluid van de stilte,Inez Weski,"Persoonlijk relaas over rechtspraak, media-aan...",Boek,3F+,NaN,NaN,Nederlands,200.0,617.0,NaN,MBO,NaN,54.0,verhalen,Leeswereld,NaN,Papier,CAT-00068
68,De zaak Ridouan T.,Hans Werdmolder,Non-fictieboek over georganiseerde criminalite...,Boek,3F,fictie,NaN,NL,143.0,230.0,NaN,MBO/NT2,Aangevuld (deels gesimuleerd),54.2,verhalen,Leeswereld,2025,papier,CAT-00069


In [15]:
lege_niveaus = df[
    df["Niveau"].isna() |
    df["Niveau"].astype(str).str.strip().eq("")
]

lege_niveaus.head(10)

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,doelgroep_extra
37,Libello voor mannen,NaN,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,NaN,fictie,NaN,Nederlands,354.0,837.0,NaN,NaN,gecontroleerd,66.5,verhalen,Leeswereld,NaN,papier,CAT-00038,NaN
38,Libello voor vrouwen,NaN,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,NaN,fictie,NaN,Nederlands,153.0,248.0,NaN,NaN,Aangevuld (deels gesimuleerd),63.5,verhalen,Uitgeverij Noord,2013,Papier,CAT-00039,NT2
39,De Startkrant,NaN,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),NaN,fictie,NaN,Nederlands,311.0,100.0,NaN,NaN,Aangevuld (deels gesimuleerd),60.0,verhalen,Leeswereld,2099,Papier,CAT-00040,NaN
40,De Startweekkrant,NaN,Wekelijkse digitale krant met begrijpelijke ar...,digitale krant,NaN,fictie,NaN,Nederlands,244.0,386.0,NaN,NaN,Aangevuld (deels gesimuleerd),70.3,verhalen,Leeswereld,NaN,Digitaal,CAT-00041,NT2
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,NaN,informatief,NaN,Nederlands,2.0,7.0,https://nos.nl/artikel/2548636-tekorten-zorgpe...,NaN,Aangevuld (deels gesimuleerd),43.5,verhalen,NaN,2013,PDF,CAT-00157,NaN
157,Nieuwe generatie fysiotherapeuten valt uit na ...,Sander Zurhake (NOS),"Fysiotherapie, zorg",online artikel,NaN,informatief,NaN,Nederlands,1.0,14.0,https://nos.nl/artikel/2476626-nieuwe-generati...,NaN,Aangevuld (deels gesimuleerd),44.6,verhalen,Leeswereld,2023,online,CAT-00158,NT2


In [32]:
#Missing values in de "Niveau" kolom vullen met "Onbekend"
# Lege waarden in Niveau herkennen
lege_niveaus = (
    df["Niveau"].isna()
    | df["Niveau"].astype(str).str.strip().eq("")
)
# Alleen lege Niveau-waarden aanpassen bij genre fictie
voorwaarde = (
    lege_niveaus
    & df["genre"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("fictie")
)

df.loc[voorwaarde, "Niveau"] = "2F"
df.iloc[37:41]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,2F,fictie,NaN,Nederlands,354.0,837.0,NaN,,gecontroleerd,66.5,verhalen,Leeswereld,NaN,papier,CAT-00038
38,Libello voor vrouwen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,2F,fictie,NaN,Nederlands,153.0,248.0,NaN,NT2,Aangevuld (deels gesimuleerd),63.5,verhalen,Uitgeverij Noord,2013,Papier,CAT-00039
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),2F,fictie,NaN,Nederlands,311.0,100.0,NaN,,Aangevuld (deels gesimuleerd),60.0,verhalen,Leeswereld,2099,Papier,CAT-00040
40,De Startweekkrant,Onbekend,Wekelijkse digitale krant met begrijpelijke ar...,digitale krant,2F,fictie,NaN,Nederlands,244.0,386.0,NaN,NT2,Aangevuld (deels gesimuleerd),70.3,verhalen,Leeswereld,NaN,Digitaal,CAT-00041


In [33]:
#Missing values in de "Niveau" kolom vullen met "Onbekend"
# Lege waarden in Niveau herkennen
lege_niveaus = (
    df["Niveau"].isna()
    | df["Niveau"].astype(str).str.strip().eq("")
)
# Alleen lege Niveau-waarden aanpassen bij genre fictie
voorwaarde = (
    lege_niveaus
    & df["genre"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("informatief")
)

df.loc[voorwaarde, "Niveau"] = "3F"
df.iloc[156:158]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,3F,informatief,NaN,Nederlands,2.0,7.0,https://nos.nl/artikel/2548636-tekorten-zorgpe...,,Aangevuld (deels gesimuleerd),43.5,verhalen,NaN,2013,PDF,CAT-00157
157,Nieuwe generatie fysiotherapeuten valt uit na ...,Sander Zurhake (NOS),"Fysiotherapie, zorg",online artikel,3F,informatief,NaN,Nederlands,1.0,14.0,https://nos.nl/artikel/2476626-nieuwe-generati...,NT2,Aangevuld (deels gesimuleerd),44.6,verhalen,Leeswereld,2023,online,CAT-00158


In [34]:
lege_rijen = df[
    df["doelgroep"].isna() |
    df["doelgroep"].astype(str).str.strip().eq("")
]

lege_rijen.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,trefwoorden,taal,aantal_paginas,geschatte_leestijd_min,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,2F,fictie,NaN,Nederlands,354.0,837.0,NaN,,gecontroleerd,66.5,verhalen,Leeswereld,NaN,papier,CAT-00038
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),2F,fictie,NaN,Nederlands,311.0,100.0,NaN,,Aangevuld (deels gesimuleerd),60.0,verhalen,Leeswereld,2099,Papier,CAT-00040
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,3F,informatief,NaN,Nederlands,2.0,7.0,https://nos.nl/artikel/2548636-tekorten-zorgpe...,,Aangevuld (deels gesimuleerd),43.5,verhalen,NaN,2013,PDF,CAT-00157
